# 02 Task Dependencies and Parameters

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Build multi-task jobs: dependency graphs (DAGs) with fan-out and fan-in, <b>run-if</b> conditions, passing values between tasks with <b>task values</b>, <b>if/else</b> (condition) tasks, <b>for each</b> tasks that loop over a list, job parameters and <b>dynamic value references</b> like <code>{{job.start_time.iso_date}}</code>.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Real pipelines have several steps that depend on each other: ingest, then transform several tables in parallel, then publish. Splitting them into tasks gives you parallelism, clear failure points and the ability to repair only the failed part. Control flow (if/else, for each, run-if) is tested on the exam and appears in every orchestration interview.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Multi-task jobs</b><br>
<code>`</code><br>            ┌─▶ silver_orders ────┐<br>ingest ─────┤                     ├─▶ gold ─▶ has_rows? ─(true)─▶ publish<br>            └─▶ silver_customers ─┘                     └(false)▶ notify_empty<br><code>`</code>
<ul><li><b>depends_on</b>: a task starts when its upstream tasks finish. Tasks without dependencies between them run in parallel</li><li><b>run_if</b>: when a task runs, based on upstream results: <code>ALL_SUCCESS</code> (default), <code>AT_LEAST_ONE_SUCCESS</code>, <code>NONE_FAILED</code>, <code>ALL_DONE</code>, <code>AT_LEAST_ONE_FAILED</code>, <code>ALL_FAILED</code></li><li><b>Task values</b>: small values a task sets (<code>dbutils.jobs.taskValues.set</code>) and downstream tasks read (<code>taskValues.get</code> or <code>{{tasks.&lt;task&gt;.values.&lt;key&gt;}}</code>)</li><li><b>If/else condition task</b>: compares two values and branches with <code>outcome = "true"</code> / <code>"false"</code></li><li><b>For each task</b>: runs a nested task once per element of a list, with limited concurrency</li><li><b>Job parameters</b> apply to all tasks. <b>Dynamic value references</b> (<code>{{job.id}}</code>, <code>{{job.run_id}}</code>, <code>{{job.start_time.iso_date}}</code>, <code>{{job.parameters.name}}</code>, <code>{{task.name}}</code>) are filled in at run time</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A marketing data job processed 12 country feeds in one notebook, sequentially, in 3 hours. If one country's file was malformed, everything failed and was re-run from scratch. Rewritten as a job with a <b>for each</b> task over the country list (concurrency 4), plus a condition task that skips publishing when no new data arrived, it ran in 50 minutes, and a bad country could be repaired alone without re-running the other 11.
</div>

## Syntax

```python
jobs.Task(task_key="gold", depends_on=[jobs.TaskDependency(task_key="silver_orders"),
                                       jobs.TaskDependency(task_key="silver_customers")],
          run_if=jobs.RunIf.ALL_SUCCESS, notebook_task=...)

dbutils.jobs.taskValues.set(key="rows", value=42)                                   # in task "check"
dbutils.jobs.taskValues.get(taskKey="check", key="rows", default=0, debugValue=0)  # downstream

jobs.Task(task_key="has_rows", depends_on=[jobs.TaskDependency(task_key="check")],
          condition_task=jobs.ConditionTask(left="{{tasks.check.values.rows}}",
                                            op=jobs.ConditionTaskOp.GREATER_THAN, right="0"))
jobs.Task(task_key="publish", depends_on=[jobs.TaskDependency(task_key="has_rows", outcome="true")], ...)

jobs.Task(task_key="per_country", for_each_task=jobs.ForEachTask(
    inputs='["IN", "UK", "AE"]', concurrency=2,
    task=jobs.Task(task_key="per_country_iteration",
                   notebook_task=jobs.NotebookTask(notebook_path=..., base_parameters={"country": "{{input}}"}))))
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the schema and (on Databricks) the SDK client and notebook helper from lesson 01.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Your user name and folder.
</div>

In [0]:
CATALOG = "workspace"
DB = f"{CATALOG}.workflows"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

import base64, json, time
from datetime import timedelta
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs, workspace

w = WorkspaceClient()
me = w.current_user.me().user_name
FOLDER = f"/Users/{me}/databricks-learning-workflows"
w.workspace.mkdirs(FOLDER)

def put_notebook(name, source, language=workspace.Language.PYTHON):
    path = f"{FOLDER}/{name}"
    w.workspace.import_(path=path, format=workspace.ImportFormat.SOURCE, language=language,
                        content=base64.b64encode(source.encode()).decode(), overwrite=True)
    return path

def state(r):
    return (r.state.result_state or r.state.life_cycle_state).value

print("user:", me, "| folder:", FOLDER)

## 1. Model a DAG and its run order

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Describes a job as a Python dictionary of tasks and dependencies, and computes which tasks can run in parallel at each step, using Python's <code>graphlib</code>.<br><br>
<b>Why it matters</b><br>This is how the Jobs scheduler thinks: a task becomes ready when all its upstream tasks are done. Seeing the "waves" makes it clear where parallelism comes from, and why a long chain of single tasks is slow.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Three waves: <code>ingest</code>, then the two Silver tasks in parallel, then <code>gold</code>.
</div>

In [0]:
from graphlib import TopologicalSorter

DAG = {"ingest": [], "silver_orders": ["ingest"], "silver_customers": ["ingest"], "gold": ["silver_orders", "silver_customers"]}

ts = TopologicalSorter(DAG)
ts.prepare()
wave = 1
while ts.is_active():
    ready = sorted(ts.get_ready())
    print(f"wave {wave}: {ready}")
    ts.done(*ready)
    wave += 1

## 2. Simulate run-if conditions

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Simulates a run where <code>silver_customers</code> fails, and shows which downstream tasks run under different <code>run_if</code> settings, including a <code>cleanup</code> task that should always run.<br><br>
<b>Why it matters</b><br>By default, a failure skips everything downstream (<code>ALL_SUCCESS</code>). <code>ALL_DONE</code> is for cleanup or notification tasks that must run whatever happens, and <code>AT_LEAST_ONE_FAILED</code> for error handlers.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>gold</code> is skipped (upstream failed). <code>cleanup</code> (<code>ALL_DONE</code>) runs. <code>alert_on_failure</code> (<code>AT_LEAST_ONE_FAILED</code>) runs.
</div>

In [0]:
RULES = {
    "ALL_SUCCESS": lambda s: all(x == "SUCCESS" for x in s),
    "ALL_DONE": lambda s: True,
    "NONE_FAILED": lambda s: "FAILED" not in s and "SUCCESS" in s,      # none failed, at least one ran
    "ALL_FAILED": lambda s: all(x == "FAILED" for x in s),
    "AT_LEAST_ONE_SUCCESS": lambda s: any(x == "SUCCESS" for x in s),
    "AT_LEAST_ONE_FAILED": lambda s: any(x == "FAILED" for x in s),
}

def simulate(tasks, failing):
    results = {}
    for task in TopologicalSorter({t: d["deps"] for t, d in tasks.items()}).static_order():
        upstream = [results[u] for u in tasks[task]["deps"]]
        if upstream and not RULES[tasks[task].get("run_if", "ALL_SUCCESS")](upstream):
            results[task] = "SKIPPED"
        else:
            results[task] = "FAILED" if task in failing else "SUCCESS"
    return results

TASKS = {
    "ingest": {"deps": []},
    "silver_orders": {"deps": ["ingest"]},
    "silver_customers": {"deps": ["ingest"]},
    "gold": {"deps": ["silver_orders", "silver_customers"]},
    "cleanup": {"deps": ["gold"], "run_if": "ALL_DONE"},
    "alert_on_failure": {"deps": ["silver_orders", "silver_customers"], "run_if": "AT_LEAST_ONE_FAILED"},
}
for task, result in simulate(TASKS, failing={"silver_customers"}).items():
    print(f"{task:<18} {TASKS[task].get('run_if', 'ALL_SUCCESS'):<20} {result}")

## 3. Task notebooks: task values and parameters

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates four small notebooks for a real multi-task job:
<ul><li><code>check_new_data</code> counts rows for the run date and sets a task value <code>rows</code></li><li><code>process</code> reads that task value and the job parameter</li><li><code>per_country</code> processes one country passed by a for each task</li><li><code>report_empty</code> runs when there's nothing to do</li></ul>
<b>Why it matters</b><br>Task values are the way tasks talk to each other: small values like counts, file names or flags. Large data is passed through tables, never through task values.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Four notebook paths.
</div>

In [0]:
SOURCE_T = f"{DB}.dag_orders"
spark.sql(f"CREATE OR REPLACE TABLE {SOURCE_T} AS SELECT * FROM VALUES "
          "(DATE'2024-11-01', 'IN', 10.0), (DATE'2024-11-01', 'UK', 20.0), (DATE'2024-11-01', 'AE', 30.0), (DATE'2024-11-02', 'IN', 5.0) "
          "AS t(order_date, country, amount)")

paths = {}
paths["check"] = put_notebook("dag_check_new_data", f'''# Databricks notebook source
dbutils.widgets.text("run_date", "2024-11-01")
run_date = dbutils.widgets.get("run_date")
rows = spark.table("{SOURCE_T}").filter(f"order_date = '{{run_date}}'").count()
dbutils.jobs.taskValues.set(key="rows", value=rows)
dbutils.notebook.exit(f"rows for {{run_date}}: {{rows}}")
''')
paths["process"] = put_notebook("dag_process", '''# Databricks notebook source
dbutils.widgets.text("run_date", "2024-11-01")
rows = dbutils.jobs.taskValues.get(taskKey="check", key="rows", default=0, debugValue=0)
dbutils.notebook.exit(f"processing {rows} rows for {dbutils.widgets.get('run_date')}")
''')
paths["per_country"] = put_notebook("dag_per_country", f'''# Databricks notebook source
dbutils.widgets.text("country", "IN")
dbutils.widgets.text("run_date", "2024-11-01")
country, run_date = dbutils.widgets.get("country"), dbutils.widgets.get("run_date")
total = spark.table("{SOURCE_T}").filter(f"order_date = '{{run_date}}' AND country = '{{country}}'").agg({{"amount": "sum"}}).first()[0]
dbutils.notebook.exit(f"{{country}}: {{total}}")
''')
paths["empty"] = put_notebook("dag_report_empty", '''# Databricks notebook source
dbutils.notebook.exit("no new data, nothing published")
''')
for k, v in paths.items():
    print(f"{k:<12} {v}")

## 4. Create a job with branching and a loop

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a job: <code>check</code> → <code>has_rows</code> (if/else on the task value) → on <b>true</b>, <code>process</code> and a <b>for each</b> over three countries; on <b>false</b>, <code>report_empty</code>. A <code>run_date</code> job parameter defaults to the run's start date with a dynamic value reference.<br><br>
<b>Why it matters</b><br>This one job uses most of the control-flow features you'll meet: dependencies, task values in a condition, branch outcomes, a loop with concurrency, and a dynamic default parameter.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A job ID and link. Open it in the UI to see the graph.
</div>

In [0]:
nb = lambda key, **params: jobs.NotebookTask(notebook_path=paths[key], base_parameters=params or None)
job = w.jobs.create(
    name="dbl_02_branching_and_loops",
    parameters=[jobs.JobParameterDefinition(name="run_date", default="{{job.start_time.iso_date}}")],
    tasks=[
        jobs.Task(task_key="check", notebook_task=nb("check")),
        jobs.Task(task_key="has_rows", depends_on=[jobs.TaskDependency(task_key="check")],
                  condition_task=jobs.ConditionTask(left="{{tasks.check.values.rows}}",
                                                    op=jobs.ConditionTaskOp.GREATER_THAN, right="0")),
        jobs.Task(task_key="process", depends_on=[jobs.TaskDependency(task_key="has_rows", outcome="true")],
                  notebook_task=nb("process")),
        jobs.Task(task_key="per_country", depends_on=[jobs.TaskDependency(task_key="has_rows", outcome="true")],
                  for_each_task=jobs.ForEachTask(
                      inputs='["IN", "UK", "AE"]', concurrency=2,
                      task=jobs.Task(task_key="per_country_iteration", notebook_task=nb("per_country", country="{{input}}")))),
        jobs.Task(task_key="report_empty", depends_on=[jobs.TaskDependency(task_key="has_rows", outcome="false")],
                  notebook_task=nb("empty")),
    ],
    max_concurrent_runs=1,
    tags={"course": "databricks-learning", "lesson": "07_02"},
)
JOB_ID = job.job_id
print("job id:", JOB_ID, "| open:", f"{w.config.host}/jobs/{JOB_ID}")

## 5. Run both branches

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs the job for 2024-11-01 (3 orders, so the <b>true</b> branch) and for 2030-01-01 (no orders, so the <b>false</b> branch), and prints each task's result and output.<br><br>
<b>Why it matters</b><br>Skipped tasks are a normal outcome of branching, not failures. The run is still successful.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
For 2024-11-01: <code>check</code>, <code>has_rows</code>, <code>process</code> and <code>per_country</code> succeed, and <code>report_empty</code> is excluded. For 2030-01-01: <code>process</code> and <code>per_country</code> are excluded, and <code>report_empty</code> succeeds.
</div>

In [0]:
for run_date in ["2024-11-01", "2030-01-01"]:
    run = w.jobs.run_now(job_id=JOB_ID, job_parameters={"run_date": run_date}).result(timeout=timedelta(minutes=30))
    print(f"\nrun_date={run_date}: {state(run)}  {run.run_page_url}")
    for t in sorted(run.tasks, key=lambda t: t.task_key):
        out = ""
        if t.notebook_task and t.state.result_state and t.state.result_state.value == "SUCCESS":
            out = w.jobs.get_run_output(run_id=t.run_id).notebook_output.result or ""
        print(f"  {t.task_key:<14} {state(t):<10} {out}")

## 6. Parameters and dynamic value references

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the most useful dynamic value references, and shows how parameter values resolve.<br><br>
<b>Why it matters</b><br>Hardcoded dates and IDs make jobs brittle. Dynamic values make outputs traceable (write <code>{{job.run_id}}</code> into an audit column) and give sensible defaults (<code>{{job.start_time.iso_date}}</code>).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A table of references and examples.
</div>

| Reference | Value |
|---|---|
| `{{job.id}}`, `{{job.name}}` | The job's ID and name |
| `{{job.run_id}}` | The current job run's ID |
| `{{job.start_time.iso_date}}` | The run's start date, e.g. `2026-10-06` |
| `{{job.parameters.run_date}}` | A job parameter's value, inside task settings |
| `{{job.trigger.type}}` | `periodic`, `one_time`, `file_arrival`, ... |
| `{{task.name}}`, `{{task.run_id}}` | The current task |
| `{{tasks.check.values.rows}}` | A task value set by task `check` |
| `{{input}}` | The current element inside a for each task |

In [0]:
def resolve(template, context):
    import re
    return re.sub(r"\{\{\s*([\w.]+)\s*\}\}", lambda m: str(context.get(m.group(1), m.group(0))), template)

ctx = {"job.id": 123, "job.run_id": 98765, "job.start_time.iso_date": "2026-10-06", "job.parameters.run_date": "2024-11-01"}
for t in ["/Volumes/workspace/workflows/files/out/{{job.start_time.iso_date}}/", "run {{job.run_id}} of job {{job.id}}",
          "processing {{job.parameters.run_date}}"]:
    print(f"{t:<62} -> {resolve(t, ctx)}")

## 7. Clean up

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Deletes the demo job, unless you set <code>KEEP_JOB = True</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>job deleted</code>.
</div>

In [0]:
KEEP_JOB = False
if not KEEP_JOB:
    w.jobs.delete(job_id=JOB_ID)
    print("job deleted")

## Under the hood

```
run starts ─▶ check ─(sets taskValue rows=3)─▶ has_rows: "3" > "0" ? ─ true ─▶ process
                                                         │               └──▶ per_country ─▶ [IN] [UK] (concurrency 2) ─▶ [AE]
                                                         └ false ─▶ report_empty
tasks not on the taken branch are marked "Excluded"; the run result is SUCCESS
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata</span> task values are stored with the run (small JSON values, a few KB), not in a table.

In [0]:
print(json.dumps({"tasks": [{"task_key": k, "depends_on": v} for k, v in DAG.items()]}, indent=1))

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: passing data between tasks with task values</b><br>
They're for small values. Pass data through tables, and pass table names or counts as task values.<br><br>
**⛔ Problem: <code>taskValues.get</code> fails when the notebook is run interactively**<br>There's no upstream task outside a job. Provide <code>debugValue</code> (and <code>default</code>).<br><br>
<b>⛔ Problem: cleanup task never runs after failures</b><br>Default <code>run_if</code> is <code>ALL_SUCCESS</code>. Use <code>ALL_DONE</code>.<br><br>
<b>⛔ Problem: one huge notebook doing every step</b><br>No parallelism, and a failure re-runs everything. Split into tasks, and repair only failed ones.<br><br>
<b>⛔ Problem: for each over hundreds of inputs with high concurrency</b><br>Each iteration is a task run with start-up overhead. Batch inputs, or loop inside one task when items are small.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How do tasks in a Databricks job share information?</b><br>
Through task values for small values (<code>dbutils.jobs.taskValues.set</code> and <code>get</code>, or <code>{{tasks.&lt;task&gt;.values.&lt;key&gt;}}</code> references in task settings), and through tables or files for data. Job parameters are shared by all tasks.<br><br>

<b>🎤 2. What does <code>run_if</code> do?</b><br>
It decides whether a task runs based on its upstream tasks' results: <code>ALL_SUCCESS</code> by default, or <code>ALL_DONE</code>, <code>NONE_FAILED</code>, <code>AT_LEAST_ONE_SUCCESS</code>, <code>AT_LEAST_ONE_FAILED</code>, <code>ALL_FAILED</code>. It's used for cleanup, alerting and tolerant fan-in.<br><br>

<b>🎤 3. How do you branch in a job?</b><br>
With an if/else condition task comparing two values, often a task value and a constant, and dependencies on its <code>true</code> or <code>false</code> outcome. Tasks on the branch not taken are excluded, and the run still succeeds.<br><br>

<b>🎤 4. When would you use a for each task?</b><br>
To run the same task for each element of a list, such as countries, tables or dates, with controlled concurrency. Each iteration can be monitored and repaired individually.<br><br>

<b>🎤 5. Why split a pipeline into multiple tasks?</b><br>
Parallelism for independent steps, isolation of failures, repairing only failed tasks, different compute per task, and a clear graph for monitoring.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A task must run after its upstream tasks finish, regardless of whether they succeeded or failed. Which <code>run_if</code> setting should it use?</b><br>
A. <code>ALL_SUCCESS</code><br>
B. <code>ALL_DONE</code><br>
C. <code>AT_LEAST_ONE_SUCCESS</code><br>
D. <code>NONE_FAILED</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. How can a task pass a row count to a downstream task in the same job run?</b><br>
A. A global temporary view<br>
B. <code>dbutils.jobs.taskValues.set</code> in the upstream task and <code>taskValues.get</code> downstream<br>
C. An environment variable<br>
D. <code>spark.conf.set</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q3. A job must run the same notebook for each of 20 regions, at most 5 at a time. Which feature fits best?</b><br>
A. 20 separate jobs<br>
B. A for each task with concurrency 5<br>
C. A continuous trigger<br>
D. A file arrival trigger<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>With <code>simulate</code>, add a task <code>publish</code> that depends on <code>gold</code> with <code>NONE_FAILED</code>, and check what happens when <code>ingest</code> fails</li><li>Compute the waves of a DAG where <code>report</code> depends on <code>gold</code> and on <code>quality_checks</code>, and <code>quality_checks</code> depends on both Silver tasks</li><li>(Databricks) Create a job with a for each task over <code>["2024-11-01", "2024-11-02"]</code> that runs <code>dag_check_new_data</code> with <code>run_date = {{input}}</code></li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. NONE_FAILED: runs only if no upstream task failed and at least one ran. Here gold was skipped, so publish is skipped too
TASKS["publish"] = {"deps": ["gold"], "run_if": "NONE_FAILED"}
print(simulate(TASKS, failing={"ingest"}))

# 2. Waves with an extra quality step
dag2 = dict(DAG, quality_checks=["silver_orders", "silver_customers"], report=["gold", "quality_checks"])
ts = TopologicalSorter(dag2); ts.prepare(); wave = 1
while ts.is_active():
    ready = sorted(ts.get_ready()); print(f"wave {wave}: {ready}"); ts.done(*ready); wave += 1

In [0]:
# 3. For each over dates
job = w.jobs.create(name="dbl_02_for_each_dates", tasks=[jobs.Task(
    task_key="per_date", for_each_task=jobs.ForEachTask(inputs='["2024-11-01", "2024-11-02"]', concurrency=2,
        task=jobs.Task(task_key="per_date_iteration", notebook_task=nb("check", run_date="{{input}}"))))])
run = w.jobs.run_now(job_id=job.job_id).result(timeout=timedelta(minutes=30))
print(state(run))
w.jobs.delete(job_id=job.job_id)

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Dependencies form a DAG. Independent tasks run in parallel</li><li><code>run_if</code> controls behavior after failures: <code>ALL_DONE</code> for cleanup, <code>AT_LEAST_ONE_FAILED</code> for alerts</li><li>Task values pass small values between tasks. Tables pass data</li><li>If/else condition tasks branch, and for each tasks loop with concurrency</li><li>Job parameters + dynamic value references (<code>{{job.start_time.iso_date}}</code>, <code>{{job.run_id}}</code>) avoid hardcoding</li></ul>
</div>

| Need | Feature |
|---|---|
| Order | `depends_on=[jobs.TaskDependency(task_key=...)]` |
| Always run | `run_if=jobs.RunIf.ALL_DONE` |
| Share a value | `dbutils.jobs.taskValues.set / get` |
| Branch | `condition_task` + `outcome="true"/"false"` |
| Loop | `for_each_task` with `inputs`, `concurrency`, `{{input}}` |
| Run date default | `{{job.start_time.iso_date}}` |

## Git commit

```
git add 07_workflows/02_task_dependencies_parameters.ipynb
git commit -m "add 07_02 task dependencies and parameters notebook"
```